# 从零实现可验证的 BPE Tokenizer：从 Unicode 契约到 merge-rank 编码

这不是只演示两次合并的玩具 Notebook。我们会实现一个**可训练、可编码、可解码、可序列化、可测试**的 byte-level BPE，并专门处理工程中最容易答错的部分：Unicode 规范化、预分词边界、带词频训练、重叠 pair、确定性 tie-break、merge-rank 编码、特殊 token、模型指纹与评测。

实现只依赖 Python 标准库，所有代码单元按顺序运行。教学模型的目标是把正确性契约讲清楚，而不是追求生产级吞吐。

## 1. 先定义需求，而不是先写 merge 循环

一个工程 tokenizer 至少要回答以下问题：

1. **输入契约**：接收 Python Unicode 字符串；普通文本先做 NFC，再按可回拼的 span 预分词。
2. **训练契约**：统计 span 的词频；BPE 不跨 span 合并；每轮选择加权频次最高的相邻 pair。
3. **确定性契约**：频次相同时按 pair 的字节序升序选择，因此相同语料与配置得到相同 merges。
4. **编码契约**：从 UTF-8 单字节符号出发，严格按训练得到的 merge rank 应用规则，不能用词表最长匹配替代。
5. **解码契约**：普通 token 拼回字节后严格 UTF-8 解码；因此结果等于规范化后的输入，而不一定等于规范化前的原始码点序列。
6. **特殊 token 契约**：特殊 token 原子化、显式授权、ID 不与普通词表冲突。
7. **发布契约**：规则、规范化配置、预分词器版本与特殊 token 一起序列化，并计算稳定指纹。

接口目标：train_bpe(span_counts) 产出有序 merges；BPEModel.encode(text) 产出整数 ID；BPEModel.decode(ids) 回到规范化文本。

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import sys
import unicodedata
from collections import Counter
from dataclasses import dataclass
from typing import Iterable, Mapping, Sequence

Symbol = tuple[int, ...]               # 一个 token 对应的一段原始字节
Pair = tuple[Symbol, Symbol]
SequenceKey = tuple[Symbol, ...]

print('Python 标准库加载完成；Unicode 数据版本：', unicodedata.unidata_version)

## 2. Unicode、character 与 byte：BPE 从哪里起步？

- **字符起点**直观，但 Unicode 字符集合开放且巨大；训练没见过的新字符需要 UNK、byte fallback 或动态扩表。
- **字节起点**固定只有 256 个基础符号，任意合法 UTF-8 文本都能表示，所以普通文本 OOV 为 0。代价是一个汉字或 emoji 初始会拆成多个字节，且单个 token 的字节未必能独立 UTF-8 解码。
- **规范化**决定哪些码点序列被视为同一个输入。例如 é 可以是一个预组字符，也可以是 e 加组合重音。这里使用 NFC；若改用 NFKC 或 casefold，搜索召回可能提高，但原始大小写、全半角等信息可能丢失。

预分词器必须满足 join(spans) == text，才能保留空格、换行和标点。本例使用标准库正则把连续空白、连续单词字符、连续其他字符分开。它只是明确且可测试的教学策略；生产系统通常需要版本化的 Unicode 规则或正则实现。

In [ ]:
_PRETOKEN_RE = re.compile(r'\s+|[\w]+|[^\w\s]+', flags=re.UNICODE)

def normalize_text(text: str, form: str | None = 'NFC', lowercase: bool = False) -> str:
    if form not in {None, 'NFC', 'NFKC'}:
        raise ValueError(f'不支持的 normalization form: {form}')
    result = unicodedata.normalize(form, text) if form else text
    return result.casefold() if lowercase else result

def pre_tokenize(text: str) -> list[str]:
    spans = _PRETOKEN_RE.findall(text)
    if ''.join(spans) != text:
        raise AssertionError('预分词器丢失或重排了字符')
    return spans

composed = 'café'
decomposed = 'cafe\u0301'
assert composed != decomposed
assert normalize_text(composed) == normalize_text(decomposed)

sample = '你好，BPE!  x = 1\n👩🏽‍💻'
spans = pre_tokenize(sample)
print('预分词 spans:', spans)
print('UTF-8 字节:', list(sample.encode('utf-8'))[:24], '...')


In [ ]:
def bytes_to_symbols(raw: bytes) -> SequenceKey:
    return tuple((byte,) for byte in raw)

def symbol_to_bytes(symbol: Symbol) -> bytes:
    return bytes(symbol)

def symbol_label(symbol: Symbol) -> str:
    raw = symbol_to_bytes(symbol)
    try:
        text = raw.decode('utf-8')
        return repr(text)
    except UnicodeDecodeError:
        return '0x' + raw.hex()

han_symbols = bytes_to_symbols('汉'.encode('utf-8'))
assert len(han_symbols) == 3
assert b''.join(symbol_to_bytes(s) for s in han_symbols).decode('utf-8') == '汉'
print('汉的初始 byte symbols:', [symbol_label(s) for s in han_symbols])

## 3. 训练数据：保留词频，而不是把去重词等权处理

若一个 span 在语料出现 1000 次，它内部的相邻 pair 应贡献 1000 倍计数。下面先规范化每篇文档，再预分词并构造 Counter。训练不跨 span 合并，因此不会意外学出跨单词或跨标点的 token；空白 span 本身仍被保留并可学习。

真实工程还应固定语料快照、采样比例、去重策略、语言配额和隐私清洗版本；否则即使代码确定，训练产物也不可复现。

In [ ]:
TRAINING_DOCUMENTS = [
    '搜索系统需要召回、排序和评估。搜索系统也需要稳定的分词。',
    'BPE learns frequent byte pairs; BPE keeps rare text representable.',
    'token token tokenizer tokenization tokenizer',
    '中文搜索、英文 search、代码 search_documents(query) 可以共存。',
    'def search_documents(query):\n    return bm25.search(query)',
    'emoji 👩🏽‍💻 👩🏽‍💻 and accents café cafe\u0301 are valid text.',
    'RAG = keyword_search + vector_search + reranker',
    '关键词搜索适合实体、编号和精确术语；向量搜索补充语义召回。',
]

NORMALIZATION_CONFIG = {'form': 'NFC', 'lowercase': False}
span_counts: Counter[str] = Counter()
for document in TRAINING_DOCUMENTS:
    normalized = normalize_text(document, **NORMALIZATION_CONFIG)
    span_counts.update(pre_tokenize(normalized))

assert span_counts['token'] == 2
assert span_counts['👩🏽‍💻'] == 2
print('不同 span 数:', len(span_counts), '总 span 次数:', sum(span_counts.values()))
print('最高频 spans:', span_counts.most_common(10))

## 4. 确定性 BPE trainer

每轮训练做三件事：

1. 对每个不同 span 的当前符号序列扫描相邻 pair，乘以该 span 的词频后累计。
2. 选择最高频 pair；若并列，选择字节 tuple 字典序最小者，形成稳定 tie-break。
3. 对所有序列从左到右做**非重叠**替换，并记录该规则的 rank。

重叠要区分“统计”和“替换”：序列 aaa 含两个相邻 (a,a)，统计频次是 2；但一次规则应用只能从左到右合并成 aa, a，而不能让中间 a 同时参与两次合并。这个边界经常是 BPE 面试题的陷阱。

In [ ]:
def count_pairs(vocabulary: Mapping[SequenceKey, int]) -> Counter[Pair]:
    counts: Counter[Pair] = Counter()
    for sequence, frequency in vocabulary.items():
        for left, right in zip(sequence, sequence[1:]):
            counts[(left, right)] += frequency
    return counts

def merge_pair_once(sequence: SequenceKey, pair: Pair) -> SequenceKey:
    left, right = pair
    merged: list[Symbol] = []
    index = 0
    while index < len(sequence):
        if index + 1 < len(sequence) and sequence[index] == left and sequence[index + 1] == right:
            merged.append(left + right)
            index += 2
        else:
            merged.append(sequence[index])
            index += 1
    return tuple(merged)

def train_bpe(
    frequencies: Mapping[str, int],
    num_merges: int,
    min_frequency: int = 2,
) -> tuple[tuple[Pair, ...], Counter[SequenceKey], list[dict[str, object]]]:
    if num_merges < 0 or min_frequency < 1:
        raise ValueError('num_merges 必须非负，min_frequency 必须至少为 1')
    vocabulary: Counter[SequenceKey] = Counter()
    for span, frequency in frequencies.items():
        if not isinstance(frequency, int) or frequency <= 0:
            raise ValueError('每个 span 的词频必须是正整数')
        vocabulary[bytes_to_symbols(span.encode('utf-8'))] += frequency

    merges: list[Pair] = []
    history: list[dict[str, object]] = []
    for rank in range(num_merges):
        counts = count_pairs(vocabulary)
        if not counts:
            break
        best_frequency = max(counts.values())
        if best_frequency < min_frequency:
            break
        # tuple[int, ...] 原生可比较；min 给出稳定、与 Counter 插入顺序无关的 tie-break。
        best_pair = min(pair for pair, frequency in counts.items() if frequency == best_frequency)

        new_vocabulary: Counter[SequenceKey] = Counter()
        for sequence, frequency in vocabulary.items():
            new_vocabulary[merge_pair_once(sequence, best_pair)] += frequency
        vocabulary = new_vocabulary
        merges.append(best_pair)
        history.append({'rank': rank, 'pair': best_pair, 'frequency': best_frequency})

    return tuple(merges), vocabulary, history

# 重叠 pair：统计两个，但一次只产生一个 aa。
a = (ord('a'),)
overlap_vocab = Counter({(a, a, a): 1})
assert count_pairs(overlap_vocab)[(a, a)] == 2
assert merge_pair_once((a, a, a), (a, a)) == (a + a, a)

# 稳定 tie-break：ab 与 ac 都出现一次，字节序较小的 ab 先合并。
tie_merges, _, _ = train_bpe({'ab': 1, 'ac': 1}, num_merges=1, min_frequency=1)
assert tie_merges[0] == ((ord('a'),), (ord('b'),))
print('重叠统计与稳定 tie-break 测试通过')

In [ ]:
merges, trained_vocabulary, training_history = train_bpe(
    span_counts,
    num_merges=120,
    min_frequency=2,
)

assert merges
assert merges == train_bpe(span_counts, num_merges=120, min_frequency=2)[0]
print('实际学习 merge 数:', len(merges))
for event in training_history[:12]:
    left, right = event['pair']
    print(f"rank={event['rank']:>2} freq={event['frequency']:>2}  {symbol_label(left)} + {symbol_label(right)} -> {symbol_label(left + right)}")

## 5. 编码必须使用 merge rank

训练产物不是一个无序词表，而是一串有先后依赖的规则。编码一个预分词 span 时：

1. 从单字节序列开始；
2. 查看当前所有相邻 pair 中哪些存在于 merge_ranks；
3. 选择 rank 最小的规则，并从左到右合并它的所有非重叠出现；
4. 重复直到没有可用规则。

教学实现每轮全扫描，便于证明；生产实现常用优先队列、链表或更紧凑的数据结构减少重复扫描。

In [ ]:
def apply_merge_ranks(sequence: SequenceKey, merge_ranks: Mapping[Pair, int]) -> SequenceKey:
    current = sequence
    while len(current) >= 2:
        candidates = [
            (merge_ranks[(left, right)], position, (left, right))
            for position, (left, right) in enumerate(zip(current, current[1:]))
            if (left, right) in merge_ranks
        ]
        if not candidates:
            break
        _, _, selected_pair = min(candidates)
        current = merge_pair_once(current, selected_pair)
    return current

class BPEModel:
    def __init__(
        self,
        merges: Sequence[Pair],
        normalization: Mapping[str, object],
        special_tokens: Sequence[str] | Mapping[str, int] = (),
    ) -> None:
        self.merges = tuple(merges)
        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        if len(self.merge_ranks) != len(self.merges):
            raise ValueError('merge 规则不能重复')
        self.normalization = dict(normalization)

        pieces: list[Symbol] = [(byte,) for byte in range(256)]
        seen = set(pieces)
        for left, right in self.merges:
            piece = left + right
            if piece not in seen:
                pieces.append(piece)
                seen.add(piece)
        self.id_to_piece = dict(enumerate(pieces))
        self.piece_to_id = {piece: token_id for token_id, piece in self.id_to_piece.items()}

        if isinstance(special_tokens, Mapping):
            self.special_tokens = dict(special_tokens)
        else:
            start = len(self.id_to_piece)
            self.special_tokens = {marker: start + i for i, marker in enumerate(special_tokens)}
        if any(not marker for marker in self.special_tokens):
            raise ValueError('特殊 token 字符串不能为空')
        special_ids = list(self.special_tokens.values())
        if len(set(special_ids)) != len(special_ids) or any(token_id in self.id_to_piece for token_id in special_ids):
            raise ValueError('特殊 token ID 重复或与普通词表冲突')
        self.id_to_special = {token_id: marker for marker, token_id in self.special_tokens.items()}

    def _encode_ordinary(self, text: str) -> list[int]:
        normalized = normalize_text(text, **self.normalization)
        ids: list[int] = []
        for span in pre_tokenize(normalized):
            symbols = bytes_to_symbols(span.encode('utf-8'))
            pieces = apply_merge_ranks(symbols, self.merge_ranks)
            ids.extend(self.piece_to_id[piece] for piece in pieces)
        return ids

    def encode(self, text: str, allowed_special: Iterable[str] = ()) -> list[int]:
        allowed = set(allowed_special)
        unknown_allowed = allowed.difference(self.special_tokens)
        if unknown_allowed:
            raise ValueError(f'请求了模型不存在的特殊 token: {sorted(unknown_allowed)}')
        if not self.special_tokens:
            return self._encode_ordinary(text)

        markers = sorted(self.special_tokens, key=lambda marker: (-len(marker), marker))
        pattern = '(' + '|'.join(re.escape(marker) for marker in markers) + ')'
        ids: list[int] = []
        for part in re.split(pattern, text):
            if not part:
                continue
            if part in self.special_tokens:
                if part not in allowed:
                    raise ValueError(f'文本含未授权特殊 token: {part}')
                ids.append(self.special_tokens[part])
            else:
                ids.extend(self._encode_ordinary(part))
        return ids

    def decode(self, token_ids: Iterable[int]) -> str:
        output: list[str] = []
        byte_buffer = bytearray()

        def flush_bytes() -> None:
            if byte_buffer:
                output.append(bytes(byte_buffer).decode('utf-8', errors='strict'))
                byte_buffer.clear()

        for token_id in token_ids:
            if token_id in self.id_to_special:
                flush_bytes()
                output.append(self.id_to_special[token_id])
            elif token_id in self.id_to_piece:
                byte_buffer.extend(self.id_to_piece[token_id])
            else:
                raise ValueError(f'未知 token ID: {token_id}')
        flush_bytes()
        return ''.join(output)

model = BPEModel(
    merges,
    NORMALIZATION_CONFIG,
    special_tokens=['<|bos|>', '<|eos|>', '<|unk|>'],
)
print('普通词表大小:', len(model.id_to_piece), '特殊 tokens:', model.special_tokens)

### 为什么“词表最长匹配”不是 BPE 编码？

设词表里同时有 ab 和 bc，且训练规则中 b+c 的 rank 早于 a+b。对 abc，BPE 先合并 b+c，结果是 a, bc；从左向右最长前缀却会选 ab, c。两者 token ID 和下游模型输入都不同。

所以只保存最终 token 字符串集合是不够的，必须保存 merge 顺序。下面构造一个最小反例。

In [ ]:
def greedy_longest_bytes(raw: bytes, vocabulary: set[bytes]) -> list[bytes]:
    result: list[bytes] = []
    position = 0
    while position < len(raw):
        candidates = [piece for piece in vocabulary if raw.startswith(piece, position)]
        if not candidates:
            raise ValueError('词表无法覆盖输入')
        selected = min(candidates, key=lambda piece: (-len(piece), piece))
        result.append(selected)
        position += len(selected)
    return result

manual_merges: tuple[Pair, ...] = (
    (((ord('b'),)), ((ord('c'),))),  # rank 0: b + c
    (((ord('a'),)), ((ord('b'),))),  # rank 1: a + b
)
manual_ranks = {pair: rank for rank, pair in enumerate(manual_merges)}
ranked = apply_merge_ranks(bytes_to_symbols(b'abc'), manual_ranks)
greedy = greedy_longest_bytes(b'abc', {b'a', b'b', b'c', b'ab', b'bc'})

assert [bytes(piece) for piece in ranked] == [b'a', b'bc']
assert greedy == [b'ab', b'c']
print('merge-rank BPE:', [bytes(piece) for piece in ranked])
print('最长词表匹配:', greedy)

## 6. decode、可逆性与规范化边界

byte-level BPE 的普通编码不会产生 OOV，因为任意 UTF-8 字节都有基础 ID。解码时必须先拼接字节再做一次严格 UTF-8 解码；不能逐 token 解码，因为一个 token 可能只是多字节字符的一部分。

这里的可逆性定义为：decode(encode(x)) == NFC(x)。若业务要求逐码点还原原始输入，就不能做有损 normalization，或必须另存原文。预分词只是合并边界，不应改变文本。

In [ ]:
MULTILINGUAL_CASES = [
    '自然语言处理让搜索更聪明。',
    'BM25 ranks documents; dense retrieval recalls semantics.',
    'def f(x):\n    return x + 1  # preserve spaces',
    '👩🏽‍💻 says: café vs cafe\u0301',
    '空格  与\t制表符\n都要保留',
    '未见字符：𠮷、🫠、العربية',
]

for text in MULTILINGUAL_CASES:
    token_ids = model.encode(text)
    decoded = model.decode(token_ids)
    expected = normalize_text(text, **NORMALIZATION_CONFIG)
    assert decoded == expected
    print(f'{len(text):>3} chars -> {len(token_ids):>3} tokens | round-trip=True | {text[:24]!r}')

## 7. Special token：原子化还不够，还要防注入

BOS、EOS、工具边界等特殊 token 不是普通文本片段。它们需要固定 ID，并在编码前作为不可拆分标记识别。更重要的是，调用方必须显式声明 allowed_special；否则用户输入恰好包含相同字符串时，可能伪造控制标记。

byte-level 普通文本本不需要 UNK；这里保留 UNK 只是展示某些下游协议如何预留 ID。特殊 token 左右形成 normalization 和 merge 屏障。

In [ ]:
special_text = '<|bos|>你好 👋<|eos|>'
try:
    model.encode(special_text)
    raise AssertionError('未授权特殊 token 本应被拒绝')
except ValueError as error:
    assert '未授权特殊 token' in str(error)

special_ids = model.encode(special_text, allowed_special={'<|bos|>', '<|eos|>'})
assert special_ids[0] == model.special_tokens['<|bos|>']
assert special_ids[-1] == model.special_tokens['<|eos|>']
assert model.decode(special_ids) == special_text
print('特殊 token ID 序列:', special_ids)

## 8. 评测：不要只报 vocabulary size

至少同时观察：

- **bytes/token**：规范化文本 UTF-8 字节数除以 token 数，越高通常表示压缩越强；但过长 token 会增大词表与稀有参数。
- **相对 byte baseline 压缩率**：1 - BPE token 数 / UTF-8 字节数。训练外语言可能接近 0。
- **byte coverage / OOV**：本实现实际统计 token pieces 覆盖的原始字节比例；byte-level 基础表使合法 UTF-8 普通文本的覆盖率应为 100%，否则属于实现错误。
- **round-trip pass rate**：必须按 normalization 契约计算，不应拿规范化前字符串误判。

生产评测还应按语言、代码、emoji、长度分桶，并测 p50/p95 延迟、吞吐、峰值内存和下游任务质量。

In [ ]:
@dataclass(frozen=True)
class TokenizerMetrics:
    name: str
    utf8_bytes: int
    tokens: int
    bytes_per_token: float
    compression_vs_bytes: float
    byte_coverage: float
    round_trip: bool

def evaluate_case(name: str, text: str, tokenizer: BPEModel) -> TokenizerMetrics:
    normalized = normalize_text(text, **tokenizer.normalization)
    raw = normalized.encode('utf-8')
    ids = tokenizer.encode(text)
    token_count = len(ids)
    represented_bytes = sum(len(tokenizer.id_to_piece[token_id]) for token_id in ids)
    return TokenizerMetrics(
        name=name,
        utf8_bytes=len(raw),
        tokens=token_count,
        bytes_per_token=(len(raw) / token_count) if token_count else 0.0,
        compression_vs_bytes=(1.0 - token_count / len(raw)) if raw else 0.0,
        byte_coverage=(represented_bytes / len(raw)) if raw else 1.0,
        round_trip=tokenizer.decode(ids) == normalized,
    )

evaluation_set = {
    '中文': '搜索系统需要稳定的分词和可靠的评估。',
    'English': 'tokenizer evaluation needs held-out documents',
    '代码': 'for doc in docs:\n    score = bm25(doc, query)',
    'emoji': '👩🏽‍💻🫠🚀 café',
}
metrics = [evaluate_case(name, text, model) for name, text in evaluation_set.items()]
assert all(item.round_trip and item.byte_coverage == 1.0 for item in metrics)
print(f"{'领域':<10} {'bytes':>7} {'tokens':>7} {'bytes/token':>12} {'压缩率':>9}")
for item in metrics:
    print(f'{item.name:<10} {item.utf8_bytes:>7} {item.tokens:>7} {item.bytes_per_token:>12.3f} {item.compression_vs_bytes:>8.1%}')

## 9. 序列化、版本与模型指纹

只存 token 到 ID 的映射不够。最小发布包应包含：格式版本、normalization 配置、预分词器标识、有序 merges、特殊 token 与 ID。将规范 JSON 做 SHA-256 可得到内容指纹，用于缓存键、模型/tokenizer 配对校验和灰度审计。

指纹说明“这些发布字段一致”，不自动证明训练语料一致。生产元数据还应加入训练代码提交、Unicode/正则引擎版本、语料快照哈希、训练参数与安全审计信息。

In [ ]:
FORMAT_VERSION = 1
PRETOKENIZER_ID = 'python-re-spans-v1'

def runtime_contract() -> dict[str, object]:
    return {
        'unicode_data_version': unicodedata.unidata_version,
        'python_major_minor': list(sys.version_info[:2]),
        'pretokenizer_pattern': _PRETOKEN_RE.pattern,
        'pretokenizer_flags': _PRETOKEN_RE.flags,
    }

def model_payload(tokenizer: BPEModel) -> dict[str, object]:
    return {
        'format': 'educational-byte-bpe',
        'format_version': FORMAT_VERSION,
        'normalization': tokenizer.normalization,
        'pretokenizer': PRETOKENIZER_ID,
        'runtime_contract': runtime_contract(),
        'merges': [
            [bytes(left).hex(), bytes(right).hex()]
            for left, right in tokenizer.merges
        ],
        'special_tokens': tokenizer.special_tokens,
    }

def serialize_model(tokenizer: BPEModel) -> tuple[str, str]:
    blob = json.dumps(model_payload(tokenizer), ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    fingerprint = hashlib.sha256(blob.encode('utf-8')).hexdigest()
    return blob, fingerprint

def load_model(blob: str) -> BPEModel:
    payload = json.loads(blob)
    if payload.get('format') != 'educational-byte-bpe' or payload.get('format_version') != FORMAT_VERSION:
        raise ValueError('不兼容的 tokenizer 格式或版本')
    if payload.get('pretokenizer') != PRETOKENIZER_ID:
        raise ValueError('未知预分词器版本')
    if payload.get('runtime_contract') != runtime_contract():
        raise ValueError('Unicode/Python/预分词运行时与模型契约不一致')
    restored_merges: list[Pair] = []
    for left_hex, right_hex in payload['merges']:
        restored_merges.append((tuple(bytes.fromhex(left_hex)), tuple(bytes.fromhex(right_hex))))
    return BPEModel(restored_merges, payload['normalization'], payload['special_tokens'])

serialized, fingerprint = serialize_model(model)
restored = load_model(serialized)
assert serialize_model(restored) == (serialized, fingerprint)
for text in MULTILINGUAL_CASES:
    assert restored.encode(text) == model.encode(text)
    assert restored.decode(restored.encode(text)) == normalize_text(text, **NORMALIZATION_CONFIG)

print('序列化字节数:', len(serialized.encode('utf-8')))
print('SHA-256 指纹:', fingerprint)

## 10. 复杂度与生产优化

设 U 为不同 span 当前符号长度之和，M 为训练 merge 轮数。本实现每轮重数所有 pair 并重写所有序列，时间上界约为 O(MU)，内存约为 O(U + P)，P 是不同 pair 数。词频只是计数权重，因此扫描的是不同 span，而非展开后的全部语料。

对长度 L 的单个 span，朴素编码每次合并都扫描当前序列，最坏可到 O(L²)。生产优化通常包括：

- 用 pair -> 受影响词/位置的倒排表做增量计数，而非每轮全量重数；
- 编码端用链表维护邻接关系、用最小堆维护 rank，只更新合并点附近候选；
- 将 merges、piece bytes 和 ID 表放入紧凑数组，减少 Python 对象与哈希开销；
- 对高频短文本做有界缓存，但缓存键必须包含 tokenizer 指纹与 normalization 配置；
- 限制输入长度、特殊 token 数量与异常 Unicode 路径，进行模糊测试和资源上限测试；
- 批处理、并行化和 SIMD 只能在保持与参考实现逐 ID 一致后再做。

小语料压缩率只证明实现会工作，不代表这个词表能泛化；生产词表需要大规模保留集与下游任务验证。

In [ ]:
def encode_span_with_work(text: str, tokenizer: BPEModel) -> dict[str, int]:
    normalized = normalize_text(text, **tokenizer.normalization)
    spans = pre_tokenize(normalized)
    scans = 0
    passes = 0
    output_pieces = 0
    for span in spans:
        current = bytes_to_symbols(span.encode('utf-8'))
        while len(current) >= 2:
            scans += len(current) - 1
            candidates = [
                (tokenizer.merge_ranks[(left, right)], position, (left, right))
                for position, (left, right) in enumerate(zip(current, current[1:]))
                if (left, right) in tokenizer.merge_ranks
            ]
            if not candidates:
                break
            _, _, pair = min(candidates)
            current = merge_pair_once(current, pair)
            passes += 1
        output_pieces += len(current)
    return {'utf8_bytes': len(normalized.encode('utf-8')), 'output_pieces': output_pieces, 'pair_scans': scans, 'merge_passes': passes}

work_report = encode_span_with_work('tokenizer tokenizer 搜索搜索 👩🏽‍💻', model)
assert work_report['output_pieces'] == len(model.encode('tokenizer tokenizer 搜索搜索 👩🏽‍💻'))
print('朴素参考编码工作量:', work_report)

## 11. 回归测试：把契约变成可执行断言

测试覆盖 normalization、预分词无损、带词频统计、重叠 pair、tie-break、训练确定性、merge-rank 反例、多领域 round-trip、特殊 token 授权、未知 ID、序列化一致性与 OOV。真实项目还应增加随机 Unicode property-based tests、跨语言黄金 ID 样例和与生产实现的 differential tests。

In [ ]:
def assert_raises(expected_exception: type[Exception], function, *args, **kwargs) -> None:
    try:
        function(*args, **kwargs)
    except expected_exception:
        return
    raise AssertionError(f'预期抛出 {expected_exception.__name__}')

def run_contract_tests() -> list[str]:
    passed: list[str] = []

    assert normalize_text('cafe\u0301') == 'café'
    passed.append('NFC normalization')

    tricky = '  中A_1\t👩🏽‍💻\n'
    assert ''.join(pre_tokenize(tricky)) == tricky
    passed.append('pre-tokenization coverage')

    weighted = Counter({bytes_to_symbols(b'ab'): 3})
    assert count_pairs(weighted)[((ord('a'),), (ord('b'),))] == 3
    passed.append('weighted pair count')

    assert count_pairs(Counter({(a, a, a): 1}))[(a, a)] == 2
    assert merge_pair_once((a, a, a), (a, a)) == (a + a, a)
    passed.append('overlapping pair semantics')

    assert train_bpe({'ab': 1, 'ac': 1}, 1, 1)[0][0] == ((97,), (98,))
    passed.append('stable tie-break')

    assert train_bpe(span_counts, 120, 2)[0] == merges
    passed.append('deterministic training')

    assert [bytes(piece) for piece in ranked] == [b'a', b'bc'] and greedy == [b'ab', b'c']
    passed.append('merge-rank is not longest-match')

    for text in MULTILINGUAL_CASES:
        assert model.decode(model.encode(text)) == normalize_text(text, **NORMALIZATION_CONFIG)
    passed.append('multilingual/code/emoji round-trip')

    assert_raises(ValueError, model.encode, '<|bos|>unsafe')
    assert model.decode(model.encode(special_text, {'<|bos|>', '<|eos|>'})) == special_text
    passed.append('special token authorization')

    assert_raises(ValueError, model.decode, [10**9])
    passed.append('unknown token ID rejection')

    restored_again = load_model(serialized)
    assert serialize_model(restored_again)[1] == fingerprint
    passed.append('serialization fingerprint')

    unseen = '𠮷🫠未知字符'
    assert model.decode(model.encode(unseen)) == unseen
    passed.append('byte-level OOV=0')

    return passed

passed_tests = run_contract_tests()
print(f'{len(passed_tests)}/{len(passed_tests)} tests passed')
for test_name in passed_tests:
    print('  ✓', test_name)

## 12. 面试总结：怎样把 BPE 讲成工程答案

可以按以下顺序回答：

1. BPE 从基础符号序列开始，反复合并全局最高频相邻 pair；在 NLP 中常从字符或 UTF-8 字节开始。
2. 训练统计必须带词频，明确预分词边界；重叠 pair 在统计时都计数，应用规则时从左到右非重叠替换。
3. 并列频次必须有稳定 tie-break，否则 merges、token ID 和模型输入不可复现。
4. 推理时按 merge rank 决策，不是拿最终词表做最长匹配；二者存在可构造的反例。
5. byte-level 基础表让普通文本 OOV 为 0；decode 要先拼字节再严格 UTF-8 解码。
6. round-trip 的右侧是规范化后的输入；若做 NFKC/casefold，就要承认原始码点信息可能丢失。
7. special token 要原子化、固定 ID、显式授权；发布包要版本化并携带指纹。
8. 线上要看压缩率、分桶 OOV/round-trip、延迟、内存与下游质量，并通过增量计数、堆/链表、紧凑存储和缓存优化。

一句话结论：**BPE 的核心产物是“有序合并程序”，而不只是一个词表。**

## 研究依据与进一步阅读

以下只列原论文、规范或官方实现，便于继续核对算法与工程差异：

- Philip Gage, 1994, A New Algorithm for Data Compression：经典 byte pair encoding 压缩算法的原始文章。
- Rico Sennrich, Barry Haddow, Alexandra Birch, [Neural Machine Translation of Rare Words with Subword Units](https://aclanthology.org/P16-1162/), ACL 2016：把 BPE 系统化用于开放词表神经机器翻译的原论文与配套软件。
- Unicode Consortium, [Unicode Standard Annex #15: Unicode Normalization Forms](https://www.unicode.org/reports/tr15/)：NFC/NFKC 等规范化形式的权威规范。
- Google, [SentencePiece 官方实现](https://github.com/google/sentencepiece)：包含 BPE/Unigram、原始句子训练及工程化模型格式。
- OpenAI, [tiktoken 官方实现](https://github.com/openai/tiktoken)：面向高吞吐 byte-level BPE 的实现，可用于研究 ranks、正则预分词与性能工程。

### 教学实现与生产 tokenizer 的明确差距

本 Notebook 为了可读性每轮全量重数 pair、编码时反复扫描，序列化格式也只是内存演示；没有并行训练、增量统计、紧凑二进制词表、Unicode 版本锁定、正则引擎兼容层、流式超长文本、模糊测试、资源限制和跨实现黄金测试。它适合作为正确性参考与面试推导，不应不经压测和安全审计直接接入生产模型。